In [1]:
# ============================================================
# 01 DATASET ANALYSIS
# WORLD RISK INDEX - PAPER ALIGNED IMPLEMENTATION
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import kruskal

warnings.filterwarnings("ignore")


# ============================================================
# 1. PATHS
# ============================================================

# This notebook is inside:
# open_lab/notebooks/
#
# Therefore ../data points to:
# open_lab/data/

BASE_DIR = os.path.abspath(
    os.path.join(os.getcwd(), "..")
)

RAW_DIR = os.path.join(
    BASE_DIR,
    "data",
    "raw"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs"
)

ANALYSIS_DIR = os.path.join(
    OUTPUT_DIR,
    "analysis"
)

PLOTS_DIR = os.path.join(
    OUTPUT_DIR,
    "plots"
)

os.makedirs(ANALYSIS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)


# ============================================================
# 2. FIND CSV FILE
# ============================================================

csv_files = [
    f for f in os.listdir(RAW_DIR)
    if f.lower().endswith(".csv")
]

print("CSV files found:")
for f in csv_files:
    print(" -", f)


if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No CSV file found in: {RAW_DIR}"
    )


# Prefer the World Risk Index trend file
preferred = [
    f for f in csv_files
    if "worldriskindex" in f.lower()
]


if preferred:
    DATA_FILE = os.path.join(
        RAW_DIR,
        preferred[0]
    )
else:
    DATA_FILE = os.path.join(
        RAW_DIR,
        csv_files[0]
    )


print("\nUsing dataset:")
print(DATA_FILE)


# ============================================================
# 3. LOAD DATA
# ============================================================

df = pd.read_csv(DATA_FILE)


print("\nDataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


# ============================================================
# 4. DISPLAY BASIC INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
for i, col in enumerate(df.columns):
    print(i, ":", col)

print("\nData types:")
display(
    df.dtypes.to_frame("dtype")
)

print("\nMissing values:")
missing_df = pd.DataFrame({
    "Column": df.columns,
    "Missing": df.isna().sum().values,
    "Missing_Percent":
        (df.isna().sum().values / len(df)) * 100
})

display(
    missing_df.sort_values(
        "Missing",
        ascending=False
    )
)


# ============================================================
# 5. DUPLICATES
# ============================================================

duplicate_count = df.duplicated().sum()

print(
    "\nDuplicate rows:",
    duplicate_count
)


# ============================================================
# 6. NUMERIC SUMMARY
# ============================================================

numeric_df = df.select_dtypes(
    include=np.number
)

print(
    "\nNumeric columns:",
    len(numeric_df.columns)
)

display(
    numeric_df.describe().T
)


# ============================================================
# 7. IDENTIFY IMPORTANT COLUMNS
# ============================================================

def find_column(
    columns,
    keywords
):

    for col in columns:

        normalized = (
            str(col)
            .lower()
            .replace("_", "")
            .replace("-", "")
            .replace(" ", "")
        )

        for keyword in keywords:

            key = (
                keyword
                .lower()
                .replace("_", "")
                .replace("-", "")
                .replace(" ", "")
            )

            if normalized == key:
                return col

    return None


YEAR_COL = find_column(
    df.columns,
    ["year"]
)


WRI_COL = find_column(
    df.columns,
    ["wri", "worldriskindex"]
)


EXPOSURE_COL = find_column(
    df.columns,
    ["exposure"]
)


VULNERABILITY_COL = find_column(
    df.columns,
    ["vulnerability"]
)


SUSCEPTIBILITY_COL = find_column(
    df.columns,
    ["susceptibility"]
)


print("\nDetected columns:")
print("Year:", YEAR_COL)
print("WRI:", WRI_COL)
print("Exposure:", EXPOSURE_COL)
print("Vulnerability:", VULNERABILITY_COL)
print("Susceptibility:", SUSCEPTIBILITY_COL)


# ============================================================
# 8. TARGET SUMMARY
# ============================================================

TARGET_COLUMNS = {
    "WRI": WRI_COL,
    "Exposure": EXPOSURE_COL,
    "Vulnerability": VULNERABILITY_COL,
    "Susceptibility": SUSCEPTIBILITY_COL
}


for target_name, column in TARGET_COLUMNS.items():

    print("\n" + "=" * 70)
    print(target_name)
    print("=" * 70)

    if column is None:
        print("Target column was not detected.")
        continue

    print("Column:", column)

    print("\nStatistics:")

    display(
        df[column].describe()
    )

    print("\nNumber of unique values:")

    print(
        df[column].nunique()
    )


# ============================================================
# 9. FIVE-LEVEL RISK CATEGORIES
# ============================================================
#
# The paper uses five risk categories:
#
# Very Low
# Low
# Medium
# High
# Very High
#
# Here we inspect how the continuous target scores
# distribute across five quantile groups.
#
# IMPORTANT:
# This analysis does NOT overwrite the original dataset.
# It is only used for validation/inspection.
# ============================================================

RISK_LABELS = [
    "Very Low",
    "Low",
    "Medium",
    "High",
    "Very High"
]


def make_five_classes(series):

    clean = series.dropna()

    try:

        categories = pd.qcut(
            clean.rank(
                method="first"
            ),
            q=5,
            labels=RISK_LABELS
        )

        return categories

    except Exception as e:

        print(
            "Could not create five classes:",
            e
        )

        return None


category_summary = []


for target_name, column in TARGET_COLUMNS.items():

    if column is None:
        continue

    categories = make_five_classes(
        df[column]
    )

    if categories is None:
        continue

    counts = (
        categories
        .value_counts()
        .reindex(RISK_LABELS)
    )

    for label, count in counts.items():

        category_summary.append({

            "Target": target_name,

            "Category": label,

            "Count": int(count),

            "Percentage":
                float(
                    count /
                    len(categories) *
                    100
                )

        })


category_summary_df = pd.DataFrame(
    category_summary
)


print("\nFive-category distribution:")
display(
    category_summary_df
)


category_summary_df.to_csv(
    os.path.join(
        ANALYSIS_DIR,
        "five_category_distribution.csv"
    ),
    index=False
)


# ============================================================
# 10. BOXPLOTS
# ============================================================

for target_name, column in TARGET_COLUMNS.items():

    if column is None:
        continue

    temp = df[
        [column]
    ].dropna()

    if len(temp) == 0:
        continue

    temp["Category"] = make_five_classes(
        temp[column]
    ).values

    plt.figure(
        figsize=(9, 6)
    )

    temp.boxplot(
        column=column,
        by="Category"
    )

    plt.title(
        f"{target_name} Distribution Across Risk Categories"
    )

    plt.suptitle("")

    plt.xlabel(
        "Risk Category"
    )

    plt.ylabel(
        target_name
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            PLOTS_DIR,
            f"{target_name.lower()}_boxplot.png"
        ),
        dpi=300
    )

    plt.show()

    plt.close()


# ============================================================
# 11. KRUSKAL-WALLIS TEST
# ============================================================
#
# Non-parametric test for whether the distributions
# differ between the five risk groups.
# ============================================================

kruskal_results = []


for target_name, column in TARGET_COLUMNS.items():

    if column is None:
        continue

    temp = df[
        [column]
    ].dropna()

    if len(temp) < 10:
        continue

    categories = make_five_classes(
        temp[column]
    )

    groups = []

    for label in RISK_LABELS:

        values = temp.loc[
            categories == label,
            column
        ]

        if len(values) > 0:
            groups.append(
                values.values
            )

    if len(groups) < 2:
        continue

    statistic, p_value = kruskal(
        *groups
    )

    kruskal_results.append({

        "Target": target_name,

        "Kruskal_Wallis_Statistic":
            statistic,

        "P_Value":
            p_value

    })


kruskal_df = pd.DataFrame(
    kruskal_results
)


print("\nKruskal-Wallis results:")
display(
    kruskal_df
)


kruskal_df.to_csv(
    os.path.join(
        ANALYSIS_DIR,
        "kruskal_wallis_results.csv"
    ),
    index=False
)


# ============================================================
# 12. YEARLY ANALYSIS
# ============================================================

if YEAR_COL is not None:

    print("\n" + "=" * 70)
    print("TEMPORAL ANALYSIS")
    print("=" * 70)

    df[YEAR_COL] = pd.to_numeric(
        df[YEAR_COL],
        errors="coerce"
    )

    yearly_counts = (
        df[YEAR_COL]
        .value_counts()
        .sort_index()
    )

    print("\nRecords per year:")
    display(
        yearly_counts.to_frame(
            "Records"
        )
    )

    yearly_counts.to_csv(
        os.path.join(
            ANALYSIS_DIR,
            "records_per_year.csv"
        )
    )


    # --------------------------------------------------------
    # Yearly mean target values
    # --------------------------------------------------------

    yearly_columns = [
        col
        for col in TARGET_COLUMNS.values()
        if col is not None
    ]

    if yearly_columns:

        yearly_means = (
            df.groupby(
                YEAR_COL
            )[yearly_columns]
            .mean()
        )

        print("\nYearly mean values:")
        display(
            yearly_means
        )

        yearly_means.to_csv(
            os.path.join(
                ANALYSIS_DIR,
                "yearly_target_means.csv"
            )
        )


# ============================================================
# 13. CORRELATION ANALYSIS
# ============================================================

if len(numeric_df.columns) > 1:

    correlation = numeric_df.corr()

    correlation.to_csv(
        os.path.join(
            ANALYSIS_DIR,
            "numeric_correlation_matrix.csv"
        )
    )

    print(
        "\nCorrelation matrix saved."
    )


# ============================================================
# 14. SAVE DATASET INFORMATION
# ============================================================

dataset_info = pd.DataFrame({

    "Property": [

        "Rows",
        "Columns",
        "Duplicate_Rows",
        "Numeric_Columns"

    ],

    "Value": [

        df.shape[0],
        df.shape[1],
        duplicate_count,
        len(numeric_df.columns)

    ]

})


dataset_info.to_csv(
    os.path.join(
        ANALYSIS_DIR,
        "dataset_summary.csv"
    ),
    index=False
)


# ============================================================
# 15. FINAL MESSAGE
# ============================================================

print("\n" + "=" * 70)
print("PHASE 1 DATASET ANALYSIS COMPLETED")
print("=" * 70)

print("\nOutputs saved in:")
print(ANALYSIS_DIR)

print("\nPlots saved in:")
print(PLOTS_DIR)

CSV files found:
 - worldriskindex-trend.csv

Using dataset:
c:\Users\mvars\Downloads\open_lab\data\raw\worldriskindex-trend.csv

Dataset loaded successfully.
Rows: 5018
Columns: 248

DATASET INFORMATION

First 5 rows:


,WRI.Country,ISO3.Code,Year,W,E,V,S,C,A,S_01,...,AI_04a_Norm,AI_04a_Base,AI_04b_Norm,AI_04b_Base,AI_04c_Norm,AI_04c_Base,AI_05a_Norm,AI_05a_Base,AI_05b_Norm,AI_05b_Base
0,Afghanistan,AFG,2000,4.18,0.25,69.83,61.97,73.22,75.05,64.24,...,100.00,24,91.54,24,100.00,5,89.84,28.99,57.66,6.29
1,Albania,ALB,2000,7.23,2.38,21.96,22.58,44.37,10.57,52.20,...,19.81,97,19.81,97,27.40,91,48.84,1232.01,0.01,0.00
2,Algeria,DZA,2000,10.28,2.65,39.85,29.57,52.73,40.58,47.78,...,40.23,86,42.01,86,42.43,78,44.09,1750.31,16.64,0.30
3,Andorra,AND,2000,0.29,0.02,4.32,2.84,2.15,13.18,22.22,...,12.85,98,27.52,95,35.78,84,31.68,5783.29,55.91,5.75
4,Angola,AGO,2000,10.98,2.18,55.35,35.29,62.14,77.32,64.30,...,84.23,31,100.00,21,85.65,10,58.17,604.63,93.27,325.00



Column names:
0 : WRI.Country
1 : ISO3.Code
2 : Year
3 : W
4 : E
5 : V
6 : S
7 : C
8 : A
9 : S_01
10 : S_02
11 : S_03
12 : S_04
13 : S_05
14 : C_01
15 : C_02
16 : C_03
17 : A_01
18 : A_02
19 : A_03
20 : EI_01
21 : EI_02
22 : EI_03
23 : EI_04
24 : EI_05
25 : EI_06
26 : EI_07
27 : SI_01
28 : SI_02
29 : SI_03
30 : SI_04
31 : SI_05
32 : SI_06
33 : SI_07
34 : SI_08
35 : SI_09
36 : SI_10
37 : SI_11
38 : SI_12
39 : SI_13
40 : SI_14
41 : CI_01
42 : CI_02
43 : CI_03
44 : CI_04
45 : CI_05
46 : CI_06
47 : CI_07
48 : AI_01
49 : AI_02
50 : AI_03
51 : AI_04
52 : EI_01a_Norm
53 : EI_01a_Base
54 : EI_01b_Norm
55 : EI_01b_Base
56 : EI_01c_Norm
57 : EI_01c_Base
58 : EI_01d_Norm
59 : EI_01d_Base
60 : EI_01e_Norm
61 : EI_01e_Base
62 : EI_01f_Norm
63 : EI_01f_Base
64 : EI_02a_Norm
65 : EI_02a_Base
66 : EI_02b_Norm
67 : EI_02b_Base
68 : EI_02c_Norm
69 : EI_02c_Base
70 : EI_02d_Norm
71 : EI_02d_Base
72 : EI_02e_Norm
73 : EI_02e_Base
74 : EI_02f_Norm
75 : EI_02f_Base
76 : EI_03a_Norm
77 : EI_03a_Base
78 : EI

,dtype
WRI.Country,str
ISO3.Code,str
Year,int64
W,float64
E,float64
...,...
AI_04c_Base,int64
AI_05a_Norm,float64
AI_05a_Base,float64
AI_05b_Norm,float64



Missing values:


,Column,Missing,Missing_Percent
0,WRI.Country,0,0.0
1,ISO3.Code,0,0.0
2,Year,0,0.0
3,W,0,0.0
4,E,0,0.0
...,...,...,...
243,AI_04c_Base,0,0.0
244,AI_05a_Norm,0,0.0
245,AI_05a_Base,0,0.0
246,AI_05b_Norm,0,0.0


... (5 outputs left out)

""



TEMPORAL ANALYSIS

Records per year:


,Records
Year,
2000,193
2001,193
2002,193
2003,193
2004,193
2005,193
2006,193
2007,193
2008,193



Correlation matrix saved.

PHASE 1 DATASET ANALYSIS COMPLETED

Outputs saved in:
c:\Users\mvars\Downloads\open_lab\outputs\analysis

Plots saved in:
c:\Users\mvars\Downloads\open_lab\outputs\plots
